# Predicting late deliveries
Protocol: `protocol.yaml` (temporal holdout, 5-split walk-forward CV, choose by mean CV ROC AUC, holdout scored once).

**Feature decision:** `claim_paid_eur` is compensation paid on a claim settled *after* delivery, so it is not known at prediction time (it is almost a copy of the target: 90% of rows with a claim are late, 0% without). It is excluded as leakage. `consignment` (identifier) and `dispatched_on` (split column) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv')=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert sha('holdout.csv')=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_rows=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_rows)==1610
train=df.drop(index=hold_rows).sort_values('dispatched_on',kind='stable')
hold=df.loc[np.sort(hold_rows)]
print(len(train),len(hold), train.dispatched_on.max(), hold.dispatched_on.min())

In [ ]:
TARGET='late_delivery'
CAT=['carrier']; NUM=['distance_km','weight_kg','parcels']
FEATURES=CAT+NUM
X,y=train[FEATURES],train[TARGET]
Xh,yh=hold[FEATURES],hold[TARGET]

Candidates are named before any is scored. Preprocessing lives inside each pipeline so it is fitted within each CV fold.

In [ ]:
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
candidates={
 'baseline_prior': make_pipeline(pre(False),DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(pre(True),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1': make_pipeline(pre(True),LogisticRegression(C=0.1,max_iter=1000)),
 'random_forest': make_pipeline(pre(False),RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1)),
 'hist_gb': make_pipeline(pre(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
print(list(candidates))

In [ ]:
cv=TimeSeriesSplit(n_splits=5)
folds=[(tr,va) for tr,va in cv.split(X)]
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(X.iloc[tr],y.iloc[tr])
        s.append(roc_auc_score(y.iloc[va],mm.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
cvdf=pd.DataFrame(cv_scores).T; cvdf['mean']=cvdf.mean(axis=1); print(cvdf.round(4))
chosen=cvdf['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X,y)
holdout_auc=roc_auc_score(yh,final.predict_proba(Xh)[:,1])
print('holdout roc_auc (scored once):',round(holdout_auc,4))

In [ ]:
pos=lambda idx: [int(i) for i in idx]  # df has a default RangeIndex, so index == 0-based row position
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[pos(X.index[tr]),pos(X.index[va])] for tr,va in folds],
 'cv_scores':{k:[float(v) for v in s] for k,s in cv_scores.items()},
 'chosen':chosen,
 'holdout_rows':pos(hold.index),
 'holdout_scores':{chosen:float(holdout_auc)},
}
json.dump(evidence,open('evidence.json','w'))